In [50]:
from datetime import date
import polars as pl
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
test_date = date(2026,3,1)
df_sfdps = get_sfdps_flights_day(test_date)
df_adsbx = get_adsbx_flights_for_day(test_date)
# df_model = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')
df_model = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/v2/flights.parquet')
expr = pl.col("takeoff_time").dt.date() == test_date 
df_sfdps = df_sfdps.filter(expr)
# df_adsbx = df_adsbx.filter(expr)
df_model = df_model.filter(expr)
df_model = df_model.filter(pl.col("icao").is_in(df_sfdps.get_column("icao").unique().implode()))

In [51]:
from flights.flights_comparison import df_flights_comparision, df_flights_comparison_stats, flights_comparison
from datetime import datetime, time
start_dt = datetime.combine(test_date, time.min)
end_dt = datetime.combine(test_date, time.max)
df = df_flights_comparision(df_model, df_sfdps, start_dt=start_dt, end_dt=end_dt, compare_airports=False)
stats = df_flights_comparison_stats(df)
print(stats)

{'true_positive': 26110, 'false_positive': 8645, 'false_negative': 944, 'precision': 0.7512588116817724, 'recall': 0.9651068233902566, 'f1': 0.8448607807924413, 'airport_ident_match_pct': 0.8998851014936806}


In [42]:
COLUMNS = ["icao", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "takeoff_time_df1", "takeoff_airport_ident_df1", "landing_time_df1","landing_airport_ident_df1", "same_takeoff_airport_ident", "same_landing_airport_ident", "same_airport_ident", "match_status"]
df = df.select(COLUMNS)

In [52]:
print(len(df.filter(pl.col("match_status") == "both").filter(pl.col("same_takeoff_airport_ident") == False)))
print(len(df.filter(pl.col("match_status") == "both").filter(pl.col("same_landing_airport_ident") == False)))

1163
2106


In [49]:
df.filter(pl.col("match_status") == "both").filter(pl.col("same_takeoff_airport_ident") == False)

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df1,takeoff_airport_ident_df1,landing_time_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],str,datetime[ms],str,bool,bool,bool,str
"""0a808c""",2026-03-01 15:03:37.576,null,2026-03-01 15:38:02.250,"""KFLL""",2026-03-01 15:08:00,"""MYGF""",2026-03-01 15:40:00,"""KFLL""",false,true,false,"""both"""
"""0ca2fe""",2026-03-01 14:40:54.500,null,2026-03-01 15:12:36.162,"""TUPJ""",2026-03-01 14:55:00,"""TNCM""",2026-03-01 15:16:00,"""TUPJ""",false,true,false,"""both"""
"""0d11e8""",2026-03-01 21:05:43.947,null,2026-03-01 22:29:57.783,"""KPDX""",2026-03-01 21:06:00,"""KRNT""",2026-03-01 22:34:00,"""KPDX""",false,true,false,"""both"""
"""6832c5""",2026-03-01 17:29:13.074,null,2026-03-01 21:11:58.892,"""KVCV""",2026-03-01 17:23:00,"""KRNT""",2026-03-01 21:14:00,"""KVCV""",false,true,false,"""both"""
"""a00062""",2026-03-01 01:00:43.327,"""MMTJ""",2026-03-01 01:25:13.812,"""KLGB""",2026-03-01 01:00:00,"""KSDM""",2026-03-01 01:27:00,"""KLGB""",false,true,false,"""both"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""adf6a1""",2026-03-01 20:31:27.995,"""KSTP""",2026-03-01 22:25:51.912,"""KGRB""",2026-03-01 20:39:00,"""M98""",2026-03-01 22:27:00,"""GRB""",false,false,false,"""both"""
"""adf724""",2026-03-01 15:41:43.962,"""KAEG""",2026-03-01 16:18:43.362,"""KSAF""",2026-03-01 15:56:00,"""KABQ""",2026-03-01 16:19:00,"""SAF""",false,false,false,"""both"""
"""c02387""",2026-03-01 18:31:59.001,"""K42J""",2026-03-01 20:08:28.560,"""KMLB""",2026-03-01 18:40:00,"""42J""",2026-03-01 20:02:00,"""MLB""",false,false,false,"""both"""


In [47]:
df_adsbx.filter(pl.col("icao") == "a00581")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool
"""a00581""",null,"""N100JE""",2026-03-01 15:55:30,"""KX51""",2026-03-01 17:57:00,"""KVDF""",false,false,false,false


# What am I trying to do? 
# My model and adsbx could be missing exact same flights so there's this problem here. 
# This may only be good for testing adsb matching algoirhtm. 
# Ok with BTS see if I do intersection of df_model and df_adsbx if all the flights not in bts are because bts is wrong